In [ ]:
# ============================================================
# CELL 1: Setup & Load Model (run once)
# ============================================================
import os
import time
import torch
import numpy as np
import cv2

os.environ['MUJOCO_GL'] = 'egl'

# Monkey-patch GamepadController for headless machines
from gym_hil.wrappers.intervention_utils import GamepadController
_original_update = GamepadController.update
def _safe_update(self):
    if self.controller_config is None:
        return
    _original_update(self)
GamepadController.update = _safe_update

_original_should_intervene = GamepadController.should_intervene
def _safe_should_intervene(self):
    if self.controller_config is None:
        return True
    return _original_should_intervene(self)
GamepadController.should_intervene = _safe_should_intervene

# LeRobot imports
from lerobot.configs import PreTrainedConfig
from lerobot.policies import make_policy, make_pre_post_processors
from lerobot.envs.utils import preprocess_observation
from lerobot.datasets import LeRobotDatasetMetadata
from lerobot.utils.constants import ACTION

# --- Config ---
# CHECKPOINT = "KeshavLN/deskorgv1.4_policy"
# CHECKPOINT = "lerobot/pi05_libero_finetuned_v044"
CHECKPOINT = "/teamspace/studios/this_studio/outputs/train/pi05_deskorg_v2.7/checkpoints/055000/pretrained_model"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# Load policy
policy_cfg = PreTrainedConfig.from_pretrained(CHECKPOINT)
policy_cfg.device = DEVICE
policy_cfg.pretrained_path = CHECKPOINT  # config has null — must set explicitly
policy_cfg.n_action_steps = 10   # re-predict every step instead of every 50


ds_meta = LeRobotDatasetMetadata("KeshavLN/deskorgv2.4_dataset_nv")
policy = make_policy(cfg=policy_cfg, ds_meta=ds_meta)
# when using the existing libero checkpoint, dont pass the dataset's metadata.
#from lerobot.policies.factory import get_policy_class
#policy_cls = get_policy_class(policy_cfg.type)
#policy = make_policy(cfg=policy_cfg)
#policy = policy_cls.from_pretrained(CHECKPOINT, config=policy_cfg)
policy.eval()
print("Policy loaded.")

# Load processors (compatibility shim for registry rename)
from lerobot.processor.pipeline import ProcessorStepRegistry
try:
    ProcessorStepRegistry.get("delta_actions_processor")
except KeyError:
    RelativeActionsStep = ProcessorStepRegistry.get("relative_actions_processor")
    ProcessorStepRegistry._registry["delta_actions_processor"] = RelativeActionsStep

preprocessor, postprocessor = make_pre_post_processors(
    policy_cfg=policy_cfg,
    pretrained_path=CHECKPOINT,
    preprocessor_overrides={"device_processor": {"device": DEVICE}},
)
print("Processors loaded. Ready for rollouts!")

In [ ]:
# ============================================================
# CELL 2: Run Multiple Rollouts
# ============================================================
from organizing_env import DeskOrganizerEnv
import cv2
import os
import time
import torch
import numpy as np
import mujoco

# --- Rollout settings (edit these) ---
ROLLOUTS = {
    "Put the calculator on the pad.": ("calculator", "pad", 10),
    "Put the mouse on the pad." : ("mouse", "pad", 10),
    "Put the coffee mug on the coaster." : ("mug_b", "coaster_a", 10),
    "Put the highlighter on the pad." : ("highlighter", "pad", 10)
}
MAX_STEPS = 600

# --- Setup video writer basics ---
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
fps = 30 # Default, will be updated from env

rollout_idx = 1
for task_prompt, (target_obj, dest_obj, num_episodes) in ROLLOUTS.items():
    
    # Create a safe filename for this specific task
    clean_task_name = task_prompt.replace(" ", "_").replace(".", "").lower()
    save_video_name = f"rollout_{clean_task_name}.mp4"
    task_writer = None
    
    for ep in range(num_episodes):
        print(f"\n=== Starting Rollout {rollout_idx} ===")
        print(f"Task: {task_prompt} | Target: {target_obj} | Destination: {dest_obj} | Episode: {ep+1}/{num_episodes}")
        
        env = DeskOrganizerEnv(
            task_desc=task_prompt,
            render_mode="rgb_array",
            image_obs=True,
            use_gripper=True,
            control_mode="manual",
            target_object=target_obj,
            destination_object=dest_obj,
            horizon=MAX_STEPS
        )
        
        policy.reset()
        obs, info = env.reset()
        
        # ---> TOGGLE SOLID DARK GREY BACKGROUND (Comment out to disable) <---
        env._env.sim.model.vis.rgba.background[:] = [0.2, 0.2, 0.2]; env._env.sim._render_context_offscreen.vopt.flags[mujoco.mjtRndFlag.mjRND_SKYBOX] = 0
        
        # Initialize the video writer for this task if it hasn't been created yet
        if task_writer is None:
            fps = env._env.control_freq
            # Set the video writer to 1080p
            task_writer = cv2.VideoWriter(save_video_name, fourcc, fps, (1920, 1080))

        step = 0
        done = False
        
        try:
            while not done and step < MAX_STEPS:
                
                # 1. Ask the raw simulator for a 1080p render of the frontview
                high_res_frame = env._env.sim.render(camera_name="frontview", width=1920, height=1080)
                
                # OpenGL renders come out upside-down natively, so we flip it
                high_res_frame = np.flipud(high_res_frame)
                
                # Convert to BGR for OpenCV
                display_frame = cv2.cvtColor(high_res_frame, cv2.COLOR_RGB2BGR)
                
                # Add text overlay for current task (Scaled up for 1080p)
                cv2.putText(display_frame, f"Task: {task_prompt}", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)
                cv2.putText(display_frame, f"Ep: {ep+1}/{num_episodes} | Step: {step}/{MAX_STEPS}", (20, 100), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)
                task_writer.write(display_frame)
                
                obs_torch = preprocess_observation(obs)
                obs_torch['task'] = [env.task]
                
                expected_dim = policy.config.input_features["observation.state"].shape[0]
                if obs_torch["observation.state"].shape[-1] == 9 and expected_dim == 8:
                    obs_torch["observation.state"] = obs_torch["observation.state"][..., :8]
                        
                obs_torch = preprocessor(obs_torch)
                with torch.no_grad():
                    action = policy.select_action(obs_torch)
                action = postprocessor(action)
                
                if isinstance(action, dict):
                    action_tensor = action[ACTION]
                else:
                    action_tensor = action
                action_numpy = action_tensor.squeeze(0).to('cpu').numpy()
                
                obs, reward, terminated, truncated, info = env.step(action_numpy)
                done = terminated or truncated
                
                step += 1
                if step % 50 == 0:
                    print(f'Step {step}/{MAX_STEPS}')
                    
                if info.get('is_success', False):
                    print(f'\n[SUCCESS] Task accomplished in {step} steps!')
                    # pad the video with a few more seconds of the final frame
                    pad_frame = display_frame.copy()
                    # Scaled success text for 1080p
                    cv2.putText(pad_frame, "[SUCCESS]", (20, 150), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)
                    for _ in range(fps * 2):
                        task_writer.write(pad_frame)
                    break
                    
        except KeyboardInterrupt:
            print(f'\nRollout interrupted at step {step}')
            break
        finally:
            env.close()
            print(f'Episode finished at step {step}.')
        rollout_idx += 1

    # Release the video writer for this task before moving to the next task
    if task_writer:
        task_writer.release()
        print(f"\nTask completed. Video saved to {save_video_name}")

print("\nAll rollouts completed successfully!")
